    "# 4.2e — Détection d'objets from scratch : la Focal Loss\n",
    "\n",
    "[← 4.2c — Détection anchor-based from scratch](4.2c-Detection-Anchor-From-Scratch.ipynb) · [Série 04-Vision](README.md)\n",
    "\n",
    "Le notebook [4.2c](4.2c-Detection-Anchor-From-Scratch.ipynb) a construit la chaîne complète d'un détecteur **anchor-based** dont le déséquilibre pos:neg est géré par **sous-échantillonnage** (ratio 1:3). Ce notebook aborde la troisième voie, proposée par Lin et al. (RetinaNet, 2017) : **redéfinir la loss** pour qu'elle traite elle-même le déséquilibre, sans sous-échantillonnage.\n",
    "\n",
    "L'idée centrale : dans le régime extrême (≈ 1000 negatives pour 1 positive), la **binary cross-entropy** est dominée par les **easy negatives** — ceux que le classifieur classe déjà correctement avec une probabilité $\\approx 0$. Leur gradient reste faible mais leur **nombre cumulé** noie complètement le signal des positifs. La **focal loss** multiplie chaque contribution par un facteur $(1 - p_t)^\\gamma$ qui **annule** les easy examples et **préserve** les hard examples — d'où le nom.\n",
    "\n",
    "On construit ici :\n",
    "\n",
    "- la dérivation mathématique de $FL(p_t) = -\\alpha_t (1 - p_t)^\\gamma \\log p_t$ depuis la CE pondérée ;\n",
    "- une implémentation PyTorch vectorisée, sans aucune lib spécialisée ;\n",
    "- une **preuve numérique** sur le régime $1000{:}1$ que la CE sature (gradients dominés par les easy negatives) tandis que la focal loss maintient un gradient non-nul sur les hard negatives grâce au facteur $(1 - p_t)^\\gamma$ ;\n",
    "- une **comparaison d'entraînement** sur un classifieur binaire volontairement pathologique (foreground rare), pour mesurer l'écart de convergence CE vs focal ;\n",
    "- trois exercices : dérivation du gradient analytique, variante par classes, application à un mini-détecteur.\n",
    "\n",
    "Aucun `torchvision.ops.sigmoid_focal_loss` ni `focal_loss` d'une lib tierce — on l'écrit, et on l'instrumente, pour mesurer ce qu'elle fait réellement."

## 1. Le problème : easy negatives qui dominent

La **binary cross-entropy** standard est :

$$\text{BCE}(p, y) = -y \log p - (1 - y) \log(1 - p)$$

où $y \in \{0, 1\}$ est la classe cible et $p = \sigma(z)$ la probabilité prédite. Pour un easy negative ($y = 0$, $p \approx 0{,}01$), la contribution à la loss est $\approx -\log(0{,}99) \approx 0{,}01$ — très petite individuellement. Mais sur 1000 easy negatives par image, leur contribution cumulée est $\approx 10$, soit $10^4$ fois la contribution d'un positif seul ($\approx 0{,}001$). Le gradient total est dominé par les easy negatives ; le réseau n'apprend rien sur les positifs parce qu'ils sont statistiquement invisibles.

Le remède classique est **numérique** : sous-échantillonner les négatifs pour rééquilibrer (notebook 4.2c, ratio 1:3). Le remède de RetinaNet est **analytique** : redéfinir la loss pour qu'elle-même pondère moins les easy examples. C'est l'objet de la section suivante.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE.type)

## 2. Dérivation de la focal loss depuis la BCE pondérée

Définissons $p_t$ comme la probabilité **de la classe cible** :

$$p_t = \begin{cases} p & \text{si } y = 1 \\ 1 - p & \text{si } y = 0 \end{cases}$$

Avec cette convention, BCE se récrit $\text{BCE}(p_t) = -\log p_t$. La BCE pondérée par $\alpha \in [0, 1]$ est $\text{BCE}_\alpha(p_t) = -\alpha_t \log p_t$ où $\alpha_t = \alpha$ si $y = 1$ et $\alpha_t = 1 - \alpha$ sinon. C'est le cas particulier $\gamma = 0$ de la famille :

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log p_t$$

Le facteur $(1 - p_t)^\gamma$ est le **modulateur** :

- si $p_t$ est proche de 1 (example **bien classé**), $(1 - p_t)^\gamma \approx 0$ ⇒ sa contribution à la loss s'effondre ;
- si $p_t$ est proche de 0 (example **mal classé**), $(1 - p_t)^\gamma \approx 1$ ⇒ sa contribution reste pleine.

Le paramètre $\gamma$ (focusing parameter, $\gamma \geq 0$) règle l'agressivité de l'écrasement. $\gamma = 0$ redonne la BCE pondérée ; $\gamma = 2$ est le choix par défaut de RetinaNet. $\alpha_t$ équilibre le ratio pos:neg indépendamment du $\gamma$.

In [ ]:
def focal_loss(logits, targets, gamma=2.0, alpha=0.25, reduction="mean"):
    """Focal loss from scratch, vectorisée.

    logits   : (N,) ou (N, C) — logits bruts (avant sigmoid)
    targets  : (N,) ou (N, C) — 0 ou 1 (float)
    gamma    : focusing parameter (>=0, défaut 2.0 comme RetinaNet)
    alpha    : poids de la classe positive (défaut 0.25 pour pos:neg ≈ 1:3)
    reduction : "mean" | "sum" | "none"

    Implementation :
    p   = sigmoid(logits)                # probabilité prédite pour la classe 1
    p_t = p * targets + (1 - p) * (1 - targets)  # proba de la classe cible
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = - alpha_t * (1 - p_t) ** gamma * log(p_t)

    Compatible multi-classe : on aplatit logits et targets en (N*C,).
    """
    flat_logits = logits.reshape(-1)
    flat_targets = targets.reshape(-1).to(flat_logits.dtype)
    p = torch.sigmoid(flat_logits)
    p_t = p * flat_targets + (1.0 - p) * (1.0 - flat_targets)
    alpha_t = alpha * flat_targets + (1.0 - alpha) * (1.0 - flat_targets)
    eps = 1e-9
    loss = -alpha_t * (1.0 - p_t).pow(gamma) * torch.log(p_t.clamp(min=eps))
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss


# Vérifications : cas limites
# 1) gamma=0 -> BCE pondérée par alpha
logits = torch.tensor([2.0, -2.0, 0.0])
targets = torch.tensor([1.0, 0.0, 1.0])
fl_g0 = focal_loss(logits, targets, gamma=0.0, alpha=0.5)
bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="mean")
print(f"FL(gamma=0, alpha=0.5) = {fl_g0.item():.6f}")
print(f"BCE(mean)             = {bce.item():.6f}")
print(f"écart attendu ≈ 0 :    {abs(fl_g0.item() - bce.item()) < 1e-5}")

# 2) sur un example bien classé (p_t proche de 1) la FL tend vers 0
logits = torch.tensor([5.0])         # sigmoid -> 0.993
targets = torch.tensor([1.0])        # bien classé
print(f"FL(p_t=0.993, y=1) = {focal_loss(logits, targets).item():.6e}  (quasi nul)")

# 3) sur un example mal classé (p_t proche de 0) la FL reste pleine
logits = torch.tensor([-5.0])        # sigmoid -> 0.007
targets = torch.tensor([1.0])        # mal classé
print(f"FL(p_t=0.007, y=1) = {focal_loss(logits, targets).item():.6e}  (plein régime)")

## 3. Visualisation : CE vs focal loss en fonction de $p_t$

Avant de comparer les gradients, regardons la loss elle-même. Pour $\alpha = 0{,}25$ et $\gamma \in \{0, 1, 2, 5\}$, on trace la contribution d'un example à la loss en fonction de $p_t$. Sur la classe positive ($y = 1$, donc $p_t = p$) :

- BCE ($\gamma = 0$) : la loss reste non-nulle même pour $p_t$ proche de 1 (l'easy negative côté $y = 0$ se traduit ici par $p_t$ proche de 0, mais symétriquement, la BCE reste positive partout).
- Focal avec $\gamma \geq 1$ : la loss **s'effondre** quand $p_t \to 1$, et reste significative quand $p_t$ est petit. C'est précisément ce qui rééquilibre le gradient total dans le régime déséquilibré.

À $p_t = 0{,}9$ : la CE est $\approx -0{,}25 \log(0{,}9) \approx 0{,}026$ ; la focal ($\gamma = 2$) est $\approx -0{,}25 \cdot 0{,}01 \cdot \log(0{,}9) \approx 2{,}6 \times 10^{-4}$, soit **100× plus petite**. À $p_t = 0{,}1$ (mal classé) : la focal ne réduit la loss que d'un facteur $\approx (1 - 0{,}1)^2 \approx 0{,}81$ — l'écart est marginal, c'est exactement le but.

In [ ]:
p_t = np.linspace(0.01, 0.99, 200)
alpha = 0.25
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for gamma in [0, 1, 2, 5]:
    fl = -alpha * (1 - p_t) ** gamma * np.log(p_t)
    axes[0].plot(p_t, fl, label=f"γ={gamma}")
axes[0].set_xlabel("$p_t$ (probabilité de la classe cible)")
axes[0].set_ylabel("FL")
axes[0].set_title("Focal loss vs $p_t$ (α=0.25, classe positive)")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_yscale("log")

# ratio FL / BCE : combien de fois la focal est plus petite que la CE
for gamma in [1, 2, 5]:
    ratio = (1 - p_t) ** gamma
    axes[1].plot(p_t, ratio, label=f"γ={gamma}")
axes[1].axhline(1.0, color="black", linestyle=":", label="BCE")
axes[1].set_xlabel("$p_t$")
axes[1].set_ylabel("(1 - $p_t$)$^γ$ = FL / BCE")
axes[1].set_title("Combien de fois la focal est plus petite que la BCE")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_yscale("log")
plt.tight_layout(); plt.show()

## 4. La preuve du déséquilibre : CE vs focal sur 1000:1

Construisons un mini-batch pathologique : **1 positive et 1000 negatives**, dont 950 sont des **easy negatives** ($p \approx 0{,}02$ car le classifieur les classe déjà bien comme négatives) et 50 sont des **hard negatives** ($p \approx 0{,}6$, le classifieur hésite). On regarde la **somme** des losses sur ce batch :

- en CE : $950 \times \text{CE}(p{=}0{,}02, y{=}0) \approx 950 \times 0{,}0203 = 19{,}3$ — la somme domine ; les 50 hard negatives contribuent $\approx 50 \times 0{,}36 = 18$ ; le positif $\approx 0{,}1$. Total $\approx 37{,}4$, dont **52 % vient des easy negatives** ;
- en focal ($\gamma = 2$) : les easy negatives voient leur contribution multipliée par $(1 - 0{,}98)^2 = 4 \times 10^{-4}$ — leur somme passe de 19,3 à $0{,}0077$. Les hard negatives sont multipliées par $(1 - 0{,}6)^2 = 0{,}16$ ⇒ 18 × 0,16 = 2,9. Le positif est multiplié par $(1 - 0{,}9)^2 = 0{,}01$ ⇒ 0,001 (mais il était déjà petit).

Résultat : avec la focal loss, les **hard negatives dominent** le signal — c'est exactement ce que le réseau doit apprendre. La proportion du gradient total provenant des easy negatives passe de ~52 % à <1 %.

In [ ]:
torch.manual_seed(7)
N_EASY = 950
N_HARD = 50
N_POS = 1

# logits générés : easy negatives très négatifs (p ≈ 0.02), hard negatives moyens (p ≈ 0.6), positif très positif
easy_neg_logits = torch.full((N_EASY,), -3.9)         # sigmoid(−3.9) ≈ 0.020
hard_neg_logits = torch.full((N_HARD,), 0.4)           # sigmoid(0.4) ≈ 0.598
pos_logits = torch.full((N_POS,), 2.2)                 # sigmoid(2.2) ≈ 0.901
logits = torch.cat([easy_neg_logits, hard_neg_logits, pos_logits])
targets = torch.cat([torch.zeros(N_EASY + N_HARD), torch.ones(N_POS)])

# BCE standard (alpha=1 sur les deux classes)
bce_per = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
# Focal loss gamma=2, alpha=0.5 (équilibré pour cette preuve)
fl_per = focal_loss(logits, targets, gamma=2.0, alpha=0.5, reduction="none")

bce_easy, bce_hard, bce_pos = bce_per[:N_EASY].sum(), bce_per[N_EASY:N_EASY + N_HARD].sum(), bce_per[N_EASY + N_HARD:].sum()
fl_easy, fl_hard, fl_pos = fl_per[:N_EASY].sum(), fl_per[N_EASY:N_EASY + N_HARD].sum(), fl_per[N_EASY + N_HARD:].sum()

bce_total = bce_easy + bce_hard + bce_pos
fl_total = fl_easy + fl_hard + fl_pos

print(f"BCE  total = {bce_total.item():7.3f}  | easy {bce_easy.item():7.3f} ({100*bce_easy/bce_total:5.1f}%)  hard {bce_hard.item():7.3f} ({100*bce_hard/bce_total:5.1f}%)  pos {bce_pos.item():7.3f} ({100*bce_pos/bce_total:5.1f}%)")
print(f"FL(γ=2) total = {fl_total.item():7.3f}  | easy {fl_easy.item():7.3f} ({100*fl_easy/fl_total:5.1f}%)  hard {fl_hard.item():7.3f} ({100*fl_hard/fl_total:5.1f}%)  pos {fl_pos.item():7.3f} ({100*fl_pos/fl_total:5.1f}%)")

## 5. Comparaison d'entraînement : CE vs focal sur un problème déséquilibré

Construisons un **classifieur binaire volontairement pathologique** :

- des features 2D $x \in \mathbb{R}^2$ distribuées en deux gaussiennes (classe 0 large, classe 1 petit cluster isolé) ;
- un MLP à deux couches, entraîné pendant 30 époques avec Adam ($lr = 1\mathrm{e}{-2}$) ;
- **deux entraînements** : un avec BCE standard, un avec focal ($\gamma = 2$, $\alpha = 0{,}5$) ;
- on suit la convergence de la loss et de l'accuracy par époque, ainsi que le gradient sur le dernier batch (norme, distribution entre classes).

Hypothèse falsifiable : la focal loss doit converger plus **rapidement** vers l'accuracy 1.0 sur les positifs que la BCE, qui passe l'essentiel de ses premières époques à "détasser" les easy negatives.

In [ ]:
def make_dataset(n_pos=80, n_neg=8000, seed=0):
    rng = np.random.default_rng(seed)
    X_pos = rng.normal(loc=[2.0, 2.0], scale=0.4, size=(n_pos, 2))
    X_neg = rng.normal(loc=[-1.0, -1.0], scale=1.5, size=(n_neg, 2))
    X = np.vstack([X_pos, X_neg]).astype(np.float32)
    y = np.hstack([np.ones(n_pos), np.zeros(n_neg)]).astype(np.float32)
    perm = rng.permutation(len(X))
    return torch.tensor(X[perm]), torch.tensor(y[perm])


Xtr, ytr = make_dataset(seed=1)
Xva, yva = make_dataset(seed=2)
print(f"train : {len(Xtr)} samples, {int(ytr.sum())} positifs ({100*ytr.mean():.2f}%)")
print(f"val   : {len(Xva)} samples, {int(yva.sum())} positifs ({100*yva.mean():.2f}%)")


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_one(loss_kind, epochs=30, lr=1e-2, gamma=2.0, alpha=0.5):
    torch.manual_seed(42)
    model = MLP().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist_loss, hist_acc = [], []
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        logits = model(Xtr.to(DEVICE))
        if loss_kind == "bce":
            loss = F.binary_cross_entropy_with_logits(logits, ytr.to(DEVICE))
        else:
            loss = focal_loss(logits, ytr.to(DEVICE), gamma=gamma, alpha=alpha)
        loss.backward()
        opt.step()
        hist_loss.append(float(loss.detach()))
        model.eval()
        with torch.no_grad():
            preds = (torch.sigmoid(model(Xva.to(DEVICE))) > 0.5).float()
            # accuracy sur les positifs (rappel) + accuracy globale
            tp = (preds * yva.to(DEVICE)).sum().item()
            npos = int(yva.sum().item())
            recall_pos = tp / max(npos, 1)
            acc = (preds == yva.to(DEVICE)).float().mean().item()
        hist_acc.append((acc, recall_pos))
    return hist_loss, hist_acc


t0 = time.time()
bce_loss, bce_acc = train_one("bce", epochs=30)
fl_loss, fl_acc = train_one("focal", epochs=30, gamma=2.0, alpha=0.5)
print(f"2 entraînements × 30 époques en {time.time() - t0:.1f} s")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(bce_loss, label="BCE", color="indianred")
axes[0].plot(fl_loss, label="Focal (γ=2, α=0.5)", color="seagreen")
axes[0].set_xlabel("époque"); axes[0].set_ylabel("loss train")
axes[0].set_title("Convergence de la loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([a[1] for a in bce_acc], label="BCE — rappel pos", color="indianred")
axes[1].plot([a[1] for a in fl_acc], label="Focal — rappel pos", color="seagreen")
axes[1].set_xlabel("époque"); axes[1].set_ylabel("rappel sur les positifs (val)")
axes[1].set_title("Détection des positifs — l'écart attendu")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nRappel final sur les positifs :  BCE = {bce_acc[-1][1]:.2f}   |   Focal = {fl_acc[-1][1]:.2f}")
print(f"Accuracy globale finale       :  BCE = {bce_acc[-1][0]:.3f}   |   Focal = {fl_acc[-1][0]:.3f}")

## 6. Dérivation analytique du gradient

Pour comprendre pourquoi la focal loss modifie le paysage du gradient, dérivons $\partial \text{FL} / \partial z$ où $z$ est le logit (avant sigmoid). Posons $p = \sigma(z)$, $p_t$ comme défini §2, et $\alpha_t$ idem. La dérivation passe par :

1. $\partial \text{FL} / \partial p_t = -\alpha_t (1 - p_t)^{\gamma - 1} \left[ \gamma p_t \log p_t + (p_t - 1) \right] / p_t$ (dérivation directe) ;
2. $\partial p_t / \partial z = \sigma'(z) \cdot (2y - 1) = p(1 - p) \cdot (2y - 1)$ (chain rule avec $\partial p / \partial z = p(1 - p)$ et la définition $p_t = y p + (1 - y)(1 - p)$) ;
3. composition : $\partial \text{FL} / \partial z = \partial \text{FL} / \partial p_t \cdot p (1 - p) \cdot (2y - 1)$.

Le facteur $(1 - p_t)^{\gamma - 1}$ dans la dérivée explique pourquoi un easy example ($p_t \to 1$, donc $(1 - p_t) \to 0$) voit son gradient s'évanouir — c'est la **dynamique** de la focal loss, complémentaire à la dynamique de la loss elle-même. C'est l'objet de l'exercice 1.

## Exercices

### Exercice 1 — Gradient analytique vs autograd

Implémentez `focal_loss_grad(z, y, gamma, alpha)` qui retourne le gradient $\partial \text{FL} / \partial z$ **analytiquement**, puis vérifiez sur 100 logits aléatoires qu'il coincide avec `torch.autograd.grad` à `1e-6` près. Le test est falsifiable : si les deux divergent, soit la dérivation est fausse, soit le graphe ne voit pas la même loss.

*Indice : partez de `focal_loss` et appelez `torch.autograd.grad(fl, z, create_graph=False)[0]` ; utilisez `torch.allclose(grad_auto, grad_analytique, atol=1e-5)`.*

In [ ]:
def focal_loss_grad(z, y, gamma=2.0, alpha=0.5):
    """Exercice 1 : gradient analytique de la focal loss par rapport au logit z.

    Retourne ∂FL/∂z de même shape que z.
    """
    # TODO etudiant
    pass


def test_grad(gamma=2.0, alpha=0.5, n=100):
    torch.manual_seed(0)
    z = torch.randn(n, requires_grad=True)
    y = (torch.rand(n) > 0.7).float()
    # gradient analytique
    g_ana = focal_loss_grad(z, y, gamma=gamma, alpha=alpha)
    # gradient autograd
    fl = focal_loss(z, y, gamma=gamma, alpha=alpha, reduction="sum")
    g_auto = torch.autograd.grad(fl, z, create_graph=False)[0]
    if g_ana is None:
        print("À implémenter")
        return
    ok = torch.allclose(g_auto, g_ana, atol=1e-5)
    print(f"γ={gamma} α={alpha} : autograd vs analytique | max |Δ| = {(g_auto - g_ana).abs().max():.2e}  |  match : {ok}")

test_grad()
print("Exercice 1 à compléter")

### Exercice 2 — Focal loss multi-classe (par classes)

La formulation multi-classe de la focal loss remplace $\alpha$ par un vecteur $\alpha_c$ (un poids par classe) et garde le modulateur $(1 - p_t)^\gamma$ sur la probabilité de la classe cible. Implémentez `focal_loss_multiclass(logits, targets, gamma, alpha_vec)` où `logits` est de shape `(N, C)` et `targets` est de shape `(N,)` (indices de classe) ou `(N, C)` (one-hot).

Vérifiez sur un dataset multi-classes déséquilibré (5 classes, ratio 100:10:5:3:1) que la focal loss multi-classe donne un meilleur rappel sur les classes rares que la cross-entropy standard.

*Indice : la formule est la même, avec $p_t = $ softmax(logits)[classe_cible]. Pour le gradient, partez de la cross-entropy classique et greffez le modulateur $(1 - p_t)^\gamma$.*

In [ ]:
def focal_loss_multiclass(logits, targets, gamma=2.0, alpha_vec=None):
    """Exercice 2 : focal loss multi-classe.

    logits   : (N, C)
    targets  : (N,) — indices entiers des classes cibles
    alpha_vec : (C,) — poids par classe (défaut : alpha uniforme = 1/C)
    """
    # TODO etudiant
    pass


print("Exercice 2 à compléter")

### Exercice 3 — Mini-détecteur avec focal loss

Reprenez le modèle `AnchorNet` du notebook [4.2c](4.2c-Detection-Anchor-From-Scratch.ipynb) (backbone + tête objectness) et remplacez sa `BCE` par `focal_loss(gamma=2, alpha=0.25)`. Ré-entraînez sur le même terrain synthétique (2000 images train, 400 val) avec un **déséquilibre délibéré** : générez 10 négatifs par image pour chaque positif, plutôt que le ratio 1:3 actuel.

Hypothèse falsifiable : à déséquilibre aggravé, la focal loss doit converger en accuracy de validation sans avoir besoin du sous-échantillonnage 1:3, et le mAP final doit être au moins aussi bon.

*Indice : importez `AnchorNet` depuis 4.2c (ou recopiez sa définition), utilisez le générateur de terrain synthétique de 4.2c, et remplacez uniquement la `image_loss`. Le notebook 4.2c fixe les hyperparamètres et le harnais de mesure (mAP VOC07 + VOC10).*

In [ ]:
print("Exercice 3 à compléter — reprendre AnchorNet du 4.2c et remplacer la BCE par focal_loss")

## Conclusion

- La **focal loss** ne change pas l'**objectif** de la classification : c'est toujours un classifieur binaire (ou multi-classe) qui apprend à discriminer foreground / background. Elle change la **pondération** par example en fonction de la difficulté prédite.
- Le modulateur $(1 - p_t)^\gamma$ a deux effets conjoints : la loss des easy examples s'effondre, et **leur gradient** aussi. Les deux se conjuguent pour libérer le signal des hard examples.
- La preuve numérique (§4) sur 1000:1 montre que la proportion du gradient total provenant des easy negatives passe de ~52 % (CE) à <1 % (FL, γ=2). C'est **exactement** ce que RetinaNet utilise pour détecter des objets sur 100 000 anchors par image avec un déséquilibre 10 000:1.
- $\alpha$ équilibre les classes, $\gamma$ équilibre la difficulté. Les deux se règlent indépendamment ; $\alpha = 0{,}25$, $\gamma = 2$ est le choix par défaut de RetinaNet pour des objets rares sur fond générique.
- Comparée à la **sous-échantillonnage** (4.2c, ratio 1:3), la focal loss **garde tous les exemples** : aucune information n'est jetée. C'est son avantage statistique ; sa limite est que les hyperparamètres $\gamma$ et $\alpha$ sont un choix à régler par dataset.

**Références** : Lin et al., *Focal Loss for Dense Object Detection*, ICCV 2017 (RetinaNet, papier fondateur) · He et al., *Mask R-CNN*, ICCV 2017 (utilise la focal loss pour la tête de classification des RoIAlign dans un cadre two-stage).